# Phase 4 - NLP pipeline for `customer_reason` (revised after review)
**Objective:** turn the free-text reason into leakage-safe numeric evidence for later phases, compare practical candidates, and measure - not assume - whether text helps the
official cost-sensitive objective.

**What the review found, and what changed in this version**

| Review finding | Verdict | Change |
|---|---|---|
| Selection by macro-F1 ignores the abusive class (abusive F1 was 0.04) | **Correct** | Selection now uses downstream **official-cost saving**, requires abusive AP above the base rate, and reports abusive AP and recall at a fixed false-block rate. Macro-F1 is shown but never used to choose (section 8) |
| Text alone cannot decide the label | Correct, already documented | Text stays supporting evidence; section 8 compares structured-only, text-only and combined; section 13 audits conflicting labels |
| Hand-off OOF was customer-grouped, not time-aware | **Correct** | All downstream text scores are now **rolling-origin**: a later month is scored only by a model fitted on strictly earlier requests, with an executable check (sections 7, 11, 14) |
| Try binary abuse detection, class weights and thresholds | Fair | Section 9: binary vs four-class, weighted vs unweighted, threshold chosen on an earlier slice and applied later |
| Preprocessing is not the main problem | Agree | Normaliser unchanged |
| Audit the labels | Fair | Section 13. Labels are **not** changed: they come from the official data |

**How to run:** keep this file in `FULL_CODE/notebooks/` with `src/r2sorter/nlp.py` and the official CSVs in `FULL_CODE/data/`. About 25 minutes.

In [1]:
import json
import platform
import sys
from pathlib import Path

for candidate_path in (Path("../src"), Path("src")):
    if candidate_path.exists():
        sys.path.insert(0, str(candidate_path.resolve()))

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, f1_score, log_loss

from r2sorter.config import load_config
from r2sorter.data import LABELS, load_all
from r2sorter.features import build_feature_table, tabular_columns
from r2sorter.nlp import (
    CLASSIFIERS,
    REPRESENTATIONS,
    TextCandidate,
    binary_abuse_scores,
    check_rolling,
    classification_report,
    feature_dictionary,
    language_group,
    recall_at_fpr,
    rolling_origin_out_of_fold,
    score_frame,
    select_by_cost,
    text_audit,
    threshold_at_fpr,
)
from r2sorter.router import DECISIONS, cost_vectors, route
from r2sorter.text import normalise_reason

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90)
SEED = 42
ABUSIVE = LABELS.index("abusive")
cfg = load_config()
data = load_all(cfg)
train, test, products = data["train"], data["test"], data["products"]
REPORTS = Path("../reports") if Path("../reports").exists() else Path("reports")
REPORTS.mkdir(exist_ok=True)
versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "lightgbm": lgb.__version__,
}
print(versions, "| seed", SEED)

{'python': '3.13.17', 'numpy': '2.5.4', 'pandas': '3.1.0', 'scikit-learn': '1.9.1', 'lightgbm': '4.7.0'} | seed 42


## 2. Text-field audit (report 4.1)

In [2]:
print("columns:", list(train.columns))
assert "customer_reason" in train and "label" in train and "customer_id" in train
assert "label" not in test.columns, "hidden-test labels must not be present"
audit = text_audit(train["customer_reason"], train["label"])
test_audit = text_audit(test["customer_reason"])
(REPORTS / "phase4_text_audit.json").write_text(
    json.dumps({"train": audit, "test_text_only": test_audit}, indent=2, default=str)
)
pd.DataFrame(
    {
        "train": pd.Series({k: v for k, v in audit.items() if not isinstance(v, dict)}),
        "test (text only)": pd.Series({k: v for k, v in test_audit.items() if not isinstance(v, dict)}),
    }
)

columns: ['return_id', 'order_id', 'customer_id', 'product_id', 'category', 'sub_category', 'size_ordered', 'order_date', 'delivery_date', 'return_request_date', 'delivery_days', 'days_since_delivery', 'order_value_inr', 'item_price_inr', 'discount_pct', 'items_in_order', 'payment_method', 'fulfilment_centre', 'customer_tenure_days', 'prior_orders_count', 'prior_return_rate', 'customer_reason', 'label']


,train,test (text only)
rows,43068.0000,16932.0000
missing,0.0000,0.0000
empty_after_strip,0.0000,0.0000
words_median,11.0000,11.0000
emoji_share,0.1247,0.1202
non_ascii_share,0.1247,0.1202
duplicate_text_share,0.1596,0.1312


In [3]:
group = language_group(train["customer_reason"])
print("language groups (wordlist heuristic, NOT a trained language identifier):")
print(group.value_counts().to_string())
print("\nlabel mix within each group:")
print(pd.crosstab(group, train["label"], normalize="index").round(3).to_string())
print(
    "\ntest-period group mix (drift check):",
    language_group(test["customer_reason"]).value_counts(normalize=True).round(3).to_dict(),
)
print("\nnon-ASCII share:", audit["non_ascii_share"], "| emoji share:", audit["emoji_share"])
(
    train.groupby("label", group_keys=False)[["label", "customer_reason"]].apply(
        lambda d: d.sample(4, random_state=SEED)
    )
).reset_index(drop=True)

language groups (wordlist heuristic, NOT a trained language identifier):
customer_reason
english_like    22747
hindi_marked    20321

label mix within each group:
label            abusive  genuine_fit_issue  genuine_quality_issue  genuine_wrong_item
customer_reason                                                                       
english_like       0.078              0.471                  0.288               0.164
hindi_marked       0.078              0.493                  0.271               0.158

test-period group mix (drift check): {'english_like': 0.52, 'hindi_marked': 0.48}

non-ASCII share: 0.1247 | emoji share: 0.1247


,label,customer_reason
0,abusive,nt good refund please
1,abusive,"Ghar pe nap ke dekha, size chart se kaafi chhota he"
2,abusive,khali dabba
3,abusive,"Namaskar, ; ghar walon ko pasand nahin aaay!!!"
4,genuine_fit_issue,too tight
5,genuine_fit_issue,"Hello supprt, Fitting is not good. Please arrange pickup."
6,genuine_fit_issue,"Length zyada h, ghutno ke neeche ja rahi h. Pese vapas kar do please."
7,genuine_fit_issue,"ITS TIGHT AND UNCOMFORTABLE IN THE SLEEVES, TRIED IT ON AS SOON AS IT CAMEEE..."
8,genuine_quality_issue,"Finishing is very bad, stitching is crooked. Was v excited for this."
9,genuine_quality_issue,🙏 Theek nahi hai - bhaii ki shaadi ke liye mangaya t


Missing or empty text is not proof of abuse. None is missing here; the pipeline still handles it (`fillna('')` gives a zero vector, and `txt_n_words == 0` marks it).

## 3. Validation boundaries (report 4.2)
* **Hidden test (Jul-Sep 2026):** never used to fit a vectorizer, vocabulary, model or threshold; only transformed at the very end.
* **Development rows:** requests before `holdout_start` (1 May 2026). **Later period:** May-June 2026.
* **Out-of-fold scores are rolling-origin** (time-aware): each month is scored by a model fitted on strictly earlier requests. The first 60 days have no earlier data and get customer-grouped OOF inside that initial window only; they are marked `grouped_initial`.

In [4]:
split_date = pd.Timestamp(cfg["validation"]["holdout_start"])
N_FOLDS = int(cfg["validation"]["text_oof_folds"])
frame = build_feature_table(train, test, products, cfg)
labelled = frame[frame["is_train"]].sort_values(["return_request_date", "return_id"]).reset_index(drop=True)
dev = labelled[labelled["return_request_date"] < split_date].reset_index(drop=True)
later = labelled[labelled["return_request_date"] >= split_date].reset_index(drop=True)
print(
    f"development {len(dev):,} rows ({dev.return_request_date.min().date()} to {dev.return_request_date.max().date()}); "
    f"later period {len(later):,} rows ({later.return_request_date.min().date()} to {later.return_request_date.max().date()}); "
    f"hidden test {len(test):,} rows - labels absent"
)

development 33,553 rows (2025-10-01 to 2026-04-30); later period 9,515 rows (2026-05-01 to 2026-06-30); hidden test 16,932 rows - labels absent


## 4. Conservative normalisation (report 4.3)
Lower-case, strip punctuation and emoji, squeeze repeated letters and spaces. No stemming, stop-word removal or translation. Negation and numbers survive (tested below); emoji and `!` are kept as separate shape features.

In [5]:
samples = [
    "Size 42 NOT fitting!! 😡",
    "Wrong item, UK 8 received",
    "Too smallllll",
    "Not working",
    "nahi aaya, paise wapas karo",
    "  extra   spaces  ",
]
assert "not" in normalise_reason("Not working").split()
assert "42" in normalise_reason("Size 42 fits").split()
assert normalise_reason("Too smallllll") == "too small"
assert normalise_reason("  a   b ") == "a b"
pd.DataFrame({"original": samples, "normalised": [normalise_reason(s) for s in samples]})

,original,normalised
0,Size 42 NOT fitting!! 😡,size 42 not fitting
1,"Wrong item, UK 8 received",wrong item uk 8 received
2,Too smallllll,too small
3,Not working,not working
4,"nahi aaya, paise wapas karo",nahi aaya paise wapas karo
5,extra spaces,extra spaces


## 5. TF-IDF and the candidate grid (report 4.5 - 4.6)
Word n-grams (1-2) for phrases; character n-grams (2-5, word-bounded) for spelling variants and transliteration; combined. Classifiers: Logistic Regression (probabilities) and Linear SVM (**decision scores, not probabilities**, named `txt_s_*`).
Vocabulary and IDF are learned only from the fitting rows of each fold or period.

In [6]:
params = {**cfg["text"], "svm_c": 0.5}
candidates = {f"{r}_{c}": TextCandidate(r, c, params, SEED) for c in CLASSIFIERS for r in REPRESENTATIONS}
print("candidates:", list(candidates))

candidates: ['word_logreg', 'char_logreg', 'combined_logreg', 'word_linear_svm', 'char_linear_svm', 'combined_linear_svm']


## 6. Rolling-origin out-of-fold scores for every candidate (time-aware)
For each month after the first 60 days, the model is fitted on **all strictly earlier development requests** and scores that month. The check below proves no row was scored by a model that saw its own date or later.

In [7]:
rolling = {}
for name, candidate in candidates.items():
    scores, kind, fit_end, seconds = rolling_origin_out_of_fold(
        candidate,
        dev["customer_reason"],
        dev["label"],
        dev["customer_id"],
        dev["return_request_date"],
        initial_days=60,
        n_folds=N_FOLDS,
    )
    report = check_rolling(kind, fit_end, dev["return_request_date"])
    rolling[name] = (scores, kind, fit_end)
    print(
        f"{name:22s} {seconds:6.1f}s  rolling rows {report['rolling_rows']:,}  initial rows {report['grouped_initial_rows']:,}  scored with future data: {report['rolling_rows_scored_with_future_data']}"
    )
is_rolling = rolling["word_logreg"][1] == "rolling"
print("\nrows scored out-of-time and used for evaluation:", int(is_rolling.sum()), "of", len(dev))

word_logreg               8.6s  rolling rows 23,756  initial rows 9,797  scored with future data: 0


char_logreg              28.4s  rolling rows 23,756  initial rows 9,797  scored with future data: 0


combined_logreg          37.3s  rolling rows 23,756  initial rows 9,797  scored with future data: 0


word_linear_svm           6.4s  rolling rows 23,756  initial rows 9,797  scored with future data: 0


char_linear_svm          22.7s  rolling rows 23,756  initial rows 9,797  scored with future data: 0


combined_linear_svm      27.7s  rolling rows 23,756  initial rows 9,797  scored with future data: 0

rows scored out-of-time and used for evaluation: 23756 of 33553


## 7. Candidate comparison - abuse metrics first
Evaluated on the rolling (out-of-time) rows only. **Macro-F1 is dominated by the three genuine classes and is shown for completeness; it is not used to choose.**
`recall@5%FPR` = share of abusive requests caught when only 5% of genuine requests are flagged by the abusive score; `AP` is compared with the abusive base rate.

In [8]:
ev = dev[is_rolling].reset_index(drop=True)
y_ab = (ev["label"] == "abusive").astype(int).to_numpy()
rows, class_reports = {}, {}
for name, (scores, _kind, _fit_end) in rolling.items():
    s = scores[is_rolling]
    class_reports[name] = r = classification_report(s, ev["label"])
    rows[name] = {
        "abusive_AP": r["abusive_ap"],
        "abusive_base_rate": r["abusive_base_rate"],
        "recall@1%FPR": recall_at_fpr(y_ab, s[:, ABUSIVE], 0.01),
        "recall@5%FPR": recall_at_fpr(y_ab, s[:, ABUSIVE], 0.05),
        "recall@10%FPR": recall_at_fpr(y_ab, s[:, ABUSIVE], 0.10),
        "macro_f1 (not used to choose)": r["macro_f1"],
        "accuracy": r["accuracy"],
    }
comparison = pd.DataFrame(rows).T.round(4)
comparison.to_csv(REPORTS / "phase4_candidate_comparison.csv")
comparison

,abusive_AP,abusive_base_rate,recall@1%FPR,recall@5%FPR,recall@10%FPR,macro_f1 (not used to choose),accuracy
word_logreg,0.1760,0.081,0.0613,0.1741,0.2656,0.7043,0.8722
char_logreg,0.1452,0.081,0.0312,0.1279,0.2313,0.6909,0.8714
combined_logreg,0.1728,0.081,0.0556,0.1772,0.2640,0.7085,0.8718
word_linear_svm,0.1762,0.081,0.0639,0.1746,0.2708,0.6985,0.8758
char_linear_svm,0.1447,0.081,0.0400,0.1372,0.2432,0.6877,0.8759
combined_linear_svm,0.1745,0.081,0.0644,0.1736,0.2692,0.7024,0.8768


The abusive average precision of every candidate is only about 1.6x the base rate, and all six are within noise of each other: **the text carries a weak abuse signal whichever representation is used.**
This is why section 9 asks whether another formulation helps, and section 10 judges candidates by what they add to the structured model under the official cost.

## 8. Binary abuse detection, class weights and thresholds (review step 2)
Four-class vs binary (abusive vs genuine), unweighted vs `class_weight='balanced'`. Chronological protocol: fit before 1 Mar 2026, score Mar-Apr (**threshold slice**: pick the score that flags 5% of genuine requests), then fit on all development rows and apply that threshold to May-June (**later period**). Weighting is an experiment, not a presumed fix.

In [9]:
cut = pd.Timestamp("2026-03-01")
early, slice_ = dev[dev["return_request_date"] < cut], dev[dev["return_request_date"] >= cut]
y_slice, y_later = (
    (slice_["label"] == "abusive").astype(int).to_numpy(),
    (later["label"] == "abusive").astype(int).to_numpy(),
)
experiments = []


def run(name, slice_score, later_score):
    threshold = threshold_at_fpr(y_slice, slice_score, 0.05)
    flagged = later_score >= threshold
    experiments.append(
        {
            "variant": name,
            "AP_threshold_slice": average_precision_score(y_slice, slice_score),
            "AP_later": average_precision_score(y_later, later_score),
            "recall@5%FPR_later (threshold from earlier slice)": float(flagged[y_later == 1].mean()),
            "realised_FPR_later": float(flagged[y_later == 0].mean()),
        }
    )


for rep in ("word", "combined"):
    four = candidates[f"{rep}_logreg"]
    run(
        f"four-class {rep}",
        four.fit(early["customer_reason"], early["label"]).scores(slice_["customer_reason"])[:, ABUSIVE],
        four.fit(dev["customer_reason"], dev["label"]).scores(later["customer_reason"])[:, ABUSIVE],
    )
    for weight in (None, "balanced"):
        run(
            f"binary {rep} weight={weight}",
            binary_abuse_scores(
                rep, weight, params, SEED, early["customer_reason"], early["label"], slice_["customer_reason"]
            ),
            binary_abuse_scores(
                rep, weight, params, SEED, dev["customer_reason"], dev["label"], later["customer_reason"]
            ),
        )
binary_table = pd.DataFrame(experiments).set_index("variant").round(4)
binary_table.to_csv(REPORTS / "phase4_binary_abuse_experiment.csv")
print(
    "abusive base rate in the later period:",
    round(float(y_later.mean()), 4),
    "| target false-block rate on genuine requests: 0.05",
)
binary_table

abusive base rate in the later period: 0.0828 | target false-block rate on genuine requests: 0.05


,AP_threshold_slice,AP_later,recall@5%FPR_later (threshold from earlier slice),realised_FPR_later
variant,,,,
four-class word,0.1847,0.1802,0.1662,0.0540
binary word weight=None,0.1845,0.1813,0.1701,0.0503
binary word weight=balanced,0.1682,0.1763,0.1802,0.0495
four-class combined,0.1810,0.1824,0.1599,0.0465
binary combined weight=None,0.1827,0.1826,0.1688,0.0514
binary combined weight=balanced,0.1702,0.1754,0.1802,0.0544


Read this table as: does any formulation lift the abusive average precision clearly above the four-class model, and does the earlier-slice threshold still give about 5% false blocks later (a check that the threshold transfers in time)?
Differences of a point or two of AP are not distinguishable on one window.

## 9. Selection by downstream value (review step 4)
Structured-only vs text-only vs combined is judged with a LightGBM on structured columns, trained before 1 May 2026 and scored on May-June 2026 (3 seeds), routed with the **official cost matrix** on raw probabilities.
Text features for the training rows are the **rolling-origin** scores from section 6; for May-June they come from a model fitted on all development rows.
Arms: **S0** structured only | **S1** + the candidate's text scores | **S4** + text scores + rule/shape indicators + text-reuse history.
Rule (fixed beforehand): choose the **simplest** candidate whose S4 cost saving is within max(seed sd, 0.005) of the best, among candidates whose abusive AP beats the base rate. Macro-F1 is not an input.

In [10]:
fit_f = dev
eval_f = later
y_fit, y_eval = (fit_f.label == "abusive").astype(int).to_numpy(), (eval_f.label == "abusive").astype(int).to_numpy()
reuse = ["hist_same_text", "hist_max_text_sim", "hist_n_similar_text"]
rules = [c for c in tabular_columns(frame) if c.startswith(("ev_", "txt_len", "txt_n_", "txt_upper", "txt_has"))]
s0 = [
    c
    for c in tabular_columns(frame)
    if c not in reuse and c not in rules and not c.startswith(("lab_", "txt_p_", "txt_s_"))
]
print(len(s0), "structured columns in S0 (no text-derived column)")
genuine_cost, abusive_cost = cost_vectors(cfg)


def realised(p, y):
    idx = np.array([DECISIONS.index(d) for d in route(p, cfg)])
    cost = np.where(y == 1, abusive_cost[idx], genuine_cost[idx]).mean()
    baseline = np.where(y == 1, abusive_cost[0], genuine_cost[0]).mean()
    return (
        1 - cost / baseline,
        int(((y == 0) & (idx == DECISIONS.index("escalate"))).sum()),
        int(((y == 1) & (idx == 0)).sum()),
    )


def lgb_run(x_fit, x_eval):
    out = []
    for seed in (0, 1, 2):
        model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=30,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=8,
            random_state=seed,
            verbose=-1,
        )
        p = model.fit(x_fit, y_fit).predict_proba(x_eval)[:, 1]
        saving, wrong_escalations, missed = realised(p, y_eval)
        out.append(
            {
                "AP": average_precision_score(y_eval, p),
                "log_loss": log_loss(y_eval, p),
                "saving": saving,
                "genuine_escalated": wrong_escalations,
                "abusive_approved": missed,
            }
        )
    return pd.DataFrame(out)


arm_rows = []
base_run = lgb_run(fit_f[s0], eval_f[s0])
arm_rows.append(
    {
        "candidate": "(none)",
        "arm": "S0 structured only",
        **base_run.mean().to_dict(),
        "saving_sd": base_run["saving"].std(),
    }
)
eval_text = {}
for name, candidate in candidates.items():
    kind_ = candidate.output_kind
    eval_text[name] = candidate.fit(fit_f["customer_reason"], fit_f["label"]).scores(eval_f["customer_reason"])
    t_fit = score_frame(rolling[name][0], fit_f.index, kind_)
    t_eval = score_frame(eval_text[name], eval_f.index, kind_)
    for arm, extra in (("S1 + text scores", []), ("S4 + text scores + rules + reuse", rules + reuse)):
        res = lgb_run(pd.concat([fit_f[s0 + extra], t_fit], axis=1), pd.concat([eval_f[s0 + extra], t_eval], axis=1))
        arm_rows.append({"candidate": name, "arm": arm, **res.mean().to_dict(), "saving_sd": res["saving"].std()})
    print("done", name)
ablation = pd.DataFrame(arm_rows).set_index(["candidate", "arm"]).round(4)
ablation.to_csv(REPORTS / "phase4_text_ablation.csv")
ablation

34 structured columns in S0 (no text-derived column)


done word_logreg


done char_logreg


done combined_logreg


done word_linear_svm


done char_linear_svm


done combined_linear_svm


AP  log_loss  saving  genuine_escalated  abusive_approved  saving_sd
candidate           arm                                                                                                       
(none)              S0 structured only                0.6658    0.1553  0.2618             0.0000          385.0000     0.0029
word_logreg         S1 + text scores                  0.7009    0.1445  0.2972             0.6667          348.0000     0.0078
                    S4 + text scores + rules + reuse  0.7385    0.1367  0.3390             1.0000          320.6667     0.0068
char_logreg         S1 + text scores                  0.6958    0.1457  0.2988             0.3333          346.6667     0.0093
                    S4 + text scores + rules + reuse  0.7314    0.1387  0.3312             1.0000          324.6667     0.0072
combined_logreg     S1 + text scores                  0.6998    0.1443  0.3040             0.3333          342.3333     0.0104
                    S4 + text scores + rules + reuse  0.7356    0.1375  0.3383             1.0000          322.3333     0.0061
word_linear_svm     S1 + text scores                  0.6967    0.1454  0.2961             0.6667          349.3333     0.0102
                    S4 + text scores + rules + reuse  0.7364    0.1371  0.3273             1.6667          322.0000     0.0037
char_linear_svm     S1 + text scores                  0.6924    0.1454  0.3033             0.0000          346.6667     0.0019
                    S4 + text scores + rules + reuse  0.7311    0.1384  0.3352             0.6667          321.3333     0.0107
combined_linear_svm S1 + text scores                  0.6999    0.1446  0.3074             0.3333          345.0000     0.0108
                    S4 + text scores + rules + reuse  0.7354    0.1373  0.3341             1.3333          320.0000     0.0125

In [11]:
s4 = (
    ablation.xs("S4 + text scores + rules + reuse", level="arm")
    .drop(columns=["AP", "log_loss"])
    .rename(columns={"saving": "cost_saving", "saving_sd": "cost_saving_sd"})
)
s4["abusive_AP"] = comparison["abusive_AP"]
s4["abusive_base_rate"] = comparison["abusive_base_rate"]
selected = select_by_cost(s4)
base = ablation.loc[("(none)", "S0 structured only")]
print("selected text candidate:", selected, "(macro-F1 was NOT used)")
print(
    f"\nstructured only      : AP {base.AP:.4f}, cost saving {base.saving:.4f}, genuine escalated {base.genuine_escalated:.1f}, abusive approved {base.abusive_approved:.0f}"
)
for arm in ("S1 + text scores", "S4 + text scores + rules + reuse"):
    r = ablation.loc[(selected, arm)]
    print(
        f"{arm:32s}: AP {r.AP:.4f}, cost saving {r.saving:.4f}, genuine escalated {r.genuine_escalated:.1f}, abusive approved {r.abusive_approved:.0f}"
    )
print(
    "\nA clear gain needs a cost-saving gain larger than the seed sd; any gain also has to be weighed against genuine customers escalated."
)
s4.round(4)

selected text candidate: word_logreg (macro-F1 was NOT used)

structured only      : AP 0.6658, cost saving 0.2618, genuine escalated 0.0, abusive approved 385
S1 + text scores                : AP 0.7009, cost saving 0.2972, genuine escalated 0.7, abusive approved 348
S4 + text scores + rules + reuse: AP 0.7385, cost saving 0.3390, genuine escalated 1.0, abusive approved 321

A clear gain needs a cost-saving gain larger than the seed sd; any gain also has to be weighed against genuine customers escalated.


,cost_saving,genuine_escalated,abusive_approved,cost_saving_sd,abusive_AP,abusive_base_rate
candidate,,,,,,
word_logreg,0.3390,1.0000,320.6667,0.0068,0.1760,0.081
char_logreg,0.3312,1.0000,324.6667,0.0072,0.1452,0.081
combined_logreg,0.3383,1.0000,322.3333,0.0061,0.1728,0.081
word_linear_svm,0.3273,1.6667,322.0000,0.0037,0.1762,0.081
char_linear_svm,0.3352,0.6667,321.3333,0.0107,0.1447,0.081
combined_linear_svm,0.3341,1.3333,320.0000,0.0125,0.1745,0.081


**Honest reading.** The text signal for abuse is weak on its own, but combined with structured behaviour it raises the cost saving. The extra saving comes with a small number of genuine customers escalated (probabilities are uncalibrated at this stage), so the combined arm is promising evidence, not a finished system. Calibration and fairness are handled in later phases.

## 10. The selected candidate in detail (report 4.8)
Out-of-time (rolling) results of the selected candidate, then the later period (fitted on all development rows), and language/style groups.

In [12]:
best = class_reports[selected]
print("confusion matrix (rows = truth), rolling out-of-time rows:")
display(best["confusion"])
display(best["per_class"])
print(
    f"abusive AP {best['abusive_ap']:.4f} vs base rate {best['abusive_base_rate']:.4f} | abusive recall at argmax {best['per_class'].loc['abusive', 'recall']:.3f}"
)
best["confusion"].to_csv(REPORTS / "phase4_confusion_matrix.csv")
best["per_class"].to_csv(REPORTS / "phase4_per_class_metrics.csv")
later_report = classification_report(eval_text[selected], later["label"])
pd.DataFrame(
    {
        "rolling out-of-time (dev)": [best["macro_f1"], best["accuracy"], best["abusive_ap"]],
        "later period (May-Jun)": [later_report["macro_f1"], later_report["accuracy"], later_report["abusive_ap"]],
    },
    index=["macro_f1", "accuracy", "abusive_AP"],
).round(4)

confusion matrix (rows = truth), rolling out-of-time rows:


,pred_abusive,pred_genuine_quality_issue,pred_genuine_fit_issue,pred_genuine_wrong_item
true_abusive,112,866,659,287
true_genuine_quality_issue,101,5994,318,68
true_genuine_fit_issue,46,303,10674,21
true_genuine_wrong_item,85,149,132,3941


,precision,recall,f1,support
abusive,0.3256,0.0582,0.0988,1924
genuine_quality_issue,0.8197,0.9249,0.8691,6481
genuine_fit_issue,0.9059,0.9665,0.9352,11044
genuine_wrong_item,0.9129,0.9150,0.9140,4307


abusive AP 0.1760 vs base rate 0.0810 | abusive recall at argmax 0.058


,rolling out-of-time (dev),later period (May-Jun)
macro_f1,0.7043,0.7018
accuracy,0.8722,0.8730
abusive_AP,0.1760,0.1802


In [13]:
scores_sel = rolling[selected][0][is_rolling]
pred = scores_sel.argmax(axis=1)
truth = ev["label"].map(LABELS.index).to_numpy()
groups_dev = language_group(ev["customer_reason"]).to_numpy()
words = ev["customer_reason"].str.split().str.len().to_numpy()
styles = {
    "english_like": groups_dev == "english_like",
    "hindi_marked": groups_dev == "hindi_marked",
    "very_short (<=3 words)": words <= 3,
    "long (>=12 words)": words >= 12,
}
rng = np.random.default_rng(SEED)
out = []
for name, mask in styles.items():
    idx = np.flatnonzero(mask)
    boots = [f1_score(truth[b], pred[b], average="macro") for b in (rng.choice(idx, len(idx)) for _ in range(100))]
    out.append(
        {
            "group": name,
            "rows": len(idx),
            "abusive_share": round(float((truth[idx] == ABUSIVE).mean()), 3),
            "macro_f1": round(f1_score(truth[idx], pred[idx], average="macro"), 4),
            "ci95_low": round(float(np.percentile(boots, 2.5)), 4),
            "ci95_high": round(float(np.percentile(boots, 97.5)), 4),
            "abusive_AP": round(average_precision_score(truth[idx] == ABUSIVE, scores_sel[idx, ABUSIVE]), 4),
        }
    )
subgroups = pd.DataFrame(out).set_index("group")
subgroups.to_csv(REPORTS / "phase4_subgroup_metrics.csv")
subgroups

,rows,abusive_share,macro_f1,ci95_low,ci95_high,abusive_AP
group,,,,,,
english_like,12554,0.079,0.7017,0.6948,0.7082,0.1633
hindi_marked,11202,0.083,0.7055,0.6999,0.7147,0.1894
very_short (<=3 words),4226,0.085,0.6809,0.6711,0.6955,0.1773
long (>=12 words),11660,0.075,0.7199,0.7123,0.7269,0.1668


In [14]:
# Candidate C: multilingual sentence embeddings. Check that a model can be obtained at all before planning an experiment.
import importlib.util
import urllib.error
import urllib.request

status = {"sentence_transformers_installed": importlib.util.find_spec("sentence_transformers") is not None}
try:
    with urllib.request.urlopen(
        "https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", timeout=10
    ) as response:
        status["model_hub_http_status"] = response.status
except (urllib.error.URLError, OSError) as error:
    status["model_hub_http_status"] = f"unreachable: {error}"
EMBEDDING_DECISION = "deferred"
print(status, "->", EMBEDDING_DECISION)

{'sentence_transformers_installed': False, 'model_hub_http_status': 'unreachable: <urlopen error Tunnel connection failed: 403 Forbidden>'} -> deferred


**Embeddings: explicitly deferred** (the model hub is not reachable from this workspace; the report says to test embeddings only if TF-IDF misses important semantic variation, and the weak abuse signal here comes from label ambiguity, not from spelling variation).

## 11. Leakage prevention and output-integrity tests (report 6)

In [15]:
checks = {}
sel_scores, sel_kind, sel_fit_end = rolling[selected]
checks["11a_rolling_check"] = check_rolling(
    sel_kind, sel_fit_end, dev["return_request_date"]
)  # no row scored with its own or later dates
checks["11b_no_gaps"] = int(np.isnan(sel_scores).any(axis=1).sum()) + int(dev["return_id"].duplicated().sum())
assert checks["11b_no_gaps"] == 0
# 11c. vectorizer fitting boundary: a token present only in later texts must not enter the earlier model's vocabulary
marker = "zzfuturetoken"
early_fit, future = (
    dev[dev["return_request_date"] < pd.Timestamp("2026-03-01")],
    dev[dev["return_request_date"] >= pd.Timestamp("2026-03-01")].copy(),
)
future["customer_reason"] = future["customer_reason"] + " " + marker
probe = candidates["word_logreg"].fit(early_fit["customer_reason"], early_fit["label"])
_ = probe.scores(future["customer_reason"])
assert marker not in probe._vectorizers[0].vocabulary_
checks["11c_future_token_in_vocabulary"] = 0
# 11d. hidden test has no labels and is not used by any fit above
assert "label" not in test.columns
checks["11d_test_labels_present"] = 0
# 11e. shuffled labels must find no signal (a leak path would keep the score above chance)
shuffled = dev["label"].sample(frac=1, random_state=SEED).reset_index(drop=True)
s_scores, s_kind, _, _ = rolling_origin_out_of_fold(
    candidates[selected], dev["customer_reason"], shuffled, dev["customer_id"], dev["return_request_date"], 60, N_FOLDS
)
s_report = classification_report(s_scores[s_kind == "rolling"], shuffled[s_kind == "rolling"])
checks["11e_shuffled_label_abusive_AP"] = round(s_report["abusive_ap"], 4)
checks["11e_abusive_base_rate"] = round(s_report["abusive_base_rate"], 4)
assert s_report["abusive_ap"] < 2 * s_report["abusive_base_rate"]
# 11f. why out-of-fold matters: in-sample scores look far better than honest ones
insample = candidates[selected].fit(dev["customer_reason"], dev["label"]).scores(dev["customer_reason"])
checks["11f_in_sample_abusive_AP"] = round(classification_report(insample, dev["label"])["abusive_ap"], 4)
checks["11f_out_of_time_abusive_AP"] = round(best["abusive_ap"], 4)
pd.Series(checks).to_frame("result")

,result
11a_rolling_check,"{'rows': 33553, 'rolling_rows': 23756, 'grouped_initial_rows': 9797, 'rows_without_kin..."
11b_no_gaps,0
11c_future_token_in_vocabulary,0
11d_test_labels_present,0
11e_shuffled_label_abusive_AP,0.0794
11e_abusive_base_rate,0.0756
11f_in_sample_abusive_AP,0.6422
11f_out_of_time_abusive_AP,0.176


## 12. Error analysis (report 7.4)
Examples from the rolling out-of-time predictions of the selected candidate (no hidden-test labels).

In [16]:
view = ev[["customer_reason", "label"]].copy()
view["predicted"] = [LABELS[i] for i in pred]
view["n_words"] = words
view["group"] = groups_dev


def show(title, mask, n=5):
    print(f"--- {title}: {int(mask.sum())} rows")
    display(view[mask].sample(min(n, int(mask.sum())), random_state=SEED)[["customer_reason", "label", "predicted"]])


show("genuine fit complaints predicted abusive", (view.label == "genuine_fit_issue") & (view.predicted == "abusive"))
show(
    "quality vs wrong-item confusion",
    view.label.isin(["genuine_quality_issue", "genuine_wrong_item"])
    & view.predicted.isin(["genuine_quality_issue", "genuine_wrong_item"])
    & (view.label != view.predicted),
)
show("abusive predicted genuine", (view.label == "abusive") & (view.predicted != "abusive"))
show("very short descriptions that were wrong", (view.n_words <= 3) & (view.label != view.predicted))
show("transliterated / Hindi-marked errors", (view.group == "hindi_marked") & (view.label != view.predicted))
show(
    "negation wording that was wrong",
    view.customer_reason.str.lower().str.contains(r"\b(?:not|no|nahi|nahin|never)\b") & (view.label != view.predicted),
)

--- genuine fit complaints predicted abusive: 46 rows


,customer_reason,label,predicted
19028,"🙏🙏 naap ki dikkat h, bhai ki shaadi ke liye mangaya tha",genuine_fit_issue,abusive
11576,Naap se chhota bheja hai. behen ki shaadi mei pehenna tha. dobara kabhi nahi mangaungi.,genuine_fit_issue,abusive
11887,jaisa socha tha waisa nai hai.,genuine_fit_issue,abusive
23027,not satisfied with the product. please take it back.,genuine_fit_issue,abusive
17085,bhut lose hai,genuine_fit_issue,abusive


--- quality vs wrong-item confusion: 217 rows


,customer_reason,label,predicted
22087,"Hello,, Not satisfied with the product, Tags are intact, pls take it baack 😐😐",genuine_wrong_item,genuine_quality_issue
23554,"Hello team,, Item was wet anddd smelling when I opened the packet",genuine_quality_issue,genuine_wrong_item
13412,"item ws wet and smelling when i opened the packet, tnx",genuine_wrong_item,genuine_quality_issue
19047,packet geela tha aur badbu aa rahi thi - label ke hisaab se hi dhoya tha??,genuine_wrong_item,genuine_quality_issue
1060,"PEHLE SE ISTEMAL KIYA HUWA LAG RAHA H, TGA BHI NAI H - JALI THEEK KARO - LOGON KO LOOT...",genuine_wrong_item,genuine_quality_issue


--- abusive predicted genuine: 1812 rows


,customer_reason,label,predicted
19649,fabric quality is very poor. Regards.,abusive,genuine_quality_issue
7912,"Piece mai defect hai, ek jagah se kata huwa hai, Picmup arrange kardo",abusive,genuine_quality_issue
23470,"Mere size ka naiiii h, ek size bada chahiye. me",abusive,genuine_fit_issue
20513,Fit is not proper,abusive,genuine_fit_issue
11490,"Hello support,, The length is too much for me, Please do the needful, Thank you 😐🙏",abusive,genuine_fit_issue


... (2 outputs left out)

--- transliterated / Hindi-marked errors: 1500 rows


,customer_reason,label,predicted
16890,Vapas karna hai.. vapas le lo,genuine_wrong_item,genuine_quality_issue
21469,Saman karabbbb hai. meherbani hogi.,genuine_quality_issue,genuine_fit_issue
5421,size chart aur actual product mein 2-3 cm ka farak he ; Specially friend ki shaadi ke ...,abusive,genuine_fit_issue
5258,"sir ji,.. jeans par rang ke dhabbe hain.. paise wapas chahie.. umeed hai jaldi hal hoga",abusive,genuine_quality_issue
5697,Achha nai lga. Refund kar do. 🙏,genuine_quality_issue,genuine_fit_issue


--- negation wording that was wrong: 860 rows


,customer_reason,label,predicted
19189,silai kachchi hai. dobara kabhi nahi mangaunga. 😤,abusive,genuine_quality_issue
16515,Product is not same as shown in the image. Colour shade is completely off from the web...,abusive,genuine_quality_issue
3480,Not satisfied with the product. Regards.,abusive,genuine_fit_issue
5426,Fit is not like the model photo at all. Please arrange pickup.,abusive,genuine_fit_issue
10769,"😢 Hello support, Not happy with this purchase. Kindly arrange pickup. Worst experience...",genuine_fit_issue,genuine_quality_issue


## 13. Label audit (review step 5)
Do the same words appear under different labels, and do abusive-labelled requests describe clear genuine problems? **Labels are not changed here**: they come from the official dataset, and relabelling would silently change the target. Findings are for the team and, if needed, the organisers.

In [17]:
norm = labelled["customer_reason"].map(normalise_reason)
n_labels = labelled.groupby(norm)["label"].nunique()
multi = n_labels[n_labels > 1]
in_multi = norm.isin(multi.index)
abusive_rows = labelled["label"] == "abusive"
genuine_texts = set(norm[~abusive_rows])
print(
    f"distinct normalised texts: {norm.nunique():,}; carrying more than one label: {len(multi):,} ({in_multi.mean():.1%} of rows)"
)
print(
    f"abusive rows whose exact normalised text also occurs under a genuine label: {norm[abusive_rows].isin(genuine_texts).mean():.1%}"
)
ev_cols = [c for c in labelled.columns if c.startswith("ev_")]
print(
    f"rows with at least one concrete-problem phrase (rule evidence): abusive {(labelled.loc[abusive_rows, ev_cols].sum(axis=1) > 0).mean():.1%} "
    f"vs genuine {(labelled.loc[~abusive_rows, ev_cols].sum(axis=1) > 0).mean():.1%}"
)
top = pd.crosstab(norm[in_multi], labelled["label"][in_multi])
top["total"] = top.sum(axis=1)
print("\nmost frequent texts that carry several labels:")
top.sort_values("total", ascending=False).head(12)

distinct normalised texts: 32,733; carrying more than one label: 516 (15.4% of rows)
abusive rows whose exact normalised text also occurs under a genuine label: 27.7%
rows with at least one concrete-problem phrase (rule evidence): abusive 82.5% vs genuine 78.3%

most frequent texts that carry several labels:


label,abusive,genuine_fit_issue,genuine_quality_issue,genuine_wrong_item,total
customer_reason,,,,,
too tight,3,94,0,0,97
colour different,21,7,69,0,97
tight fitting,3,92,0,0,95
size issue small,5,90,0,0,95
size small,7,81,0,0,88
too big,4,82,0,0,86
too small,14,68,0,0,82
fitting issue,8,73,0,0,81
too loose,15,62,0,0,77


**What this means.** Abusive-labelled requests are often worded like genuine complaints (see the shares above: some repeat a genuine text exactly, and most mention a concrete problem), so wording alone cannot separate them; the extra signal has to come from behaviour and context (history, repeats, payment, timing), which is what the structured features carry. No relabelling is proposed. If the label definition matters for the final write-up, the dataset README / organisers are the place to confirm it.

## 14. Freeze and hand off (report 4.9)
Two files for later phases, both leakage-safe by construction:
1. **Rolling-origin out-of-fold scores for every labelled training row** (`oof_kind` = `rolling` or `grouped_initial`; `fit_end` = latest request date the scoring model saw). Downstream users may drop `grouped_initial` rows (first 60 days).
2. **Hidden-test scores** from one model fitted on all labelled rows (it only *transforms* the July-September text; no labels are used).

In [18]:
kind_name = candidates[selected].output_kind
all_scores, all_kind, all_fit_end, seconds_all = rolling_origin_out_of_fold(
    candidates[selected],
    labelled["customer_reason"],
    labelled["label"],
    labelled["customer_id"],
    labelled["return_request_date"],
    60,
    N_FOLDS,
)
final_check = check_rolling(all_kind, all_fit_end, labelled["return_request_date"])
handoff_oof = pd.concat(
    [
        labelled[["return_id"]].reset_index(drop=True),
        pd.DataFrame({"oof_kind": all_kind, "fit_end": all_fit_end.to_numpy()}),
        score_frame(all_scores, pd.RangeIndex(len(labelled)), kind_name),
    ],
    axis=1,
)
final_model = candidates[selected].fit(labelled["customer_reason"], labelled["label"])
handoff_test = pd.concat(
    [
        test[["return_id"]].reset_index(drop=True),
        score_frame(final_model.scores(test["customer_reason"]), pd.RangeIndex(len(test)), kind_name),
    ],
    axis=1,
)
assert (
    handoff_oof["return_id"].is_unique
    and handoff_test["return_id"].is_unique
    and len(handoff_oof) == len(train)
    and len(handoff_test) == len(test)
)
handoff_oof.to_csv(REPORTS / "phase4_text_oof_scores.csv", index=False)
handoff_test.to_csv(REPORTS / "phase4_text_test_scores.csv", index=False)
feature_dictionary(kind_name).to_csv(REPORTS / "phase4_feature_dictionary.csv")
config_record = {
    "selected_candidate": selected,
    "selection_rule": "simplest within max(seed sd, 0.005) of best S4 cost saving; abusive AP must beat base rate; macro-F1 not used",
    "output_kind": kind_name,
    "text_params": params,
    "seed": SEED,
    "oof_strategy": "rolling-origin by month (strictly earlier data only); first 60 days grouped OOF inside the initial window, flagged grouped_initial",
    "rolling_check": final_check,
    "development_boundary": str(split_date.date()),
    "hidden_test_labels_used": False,
    "embedding_experiment": EMBEDDING_DECISION,
    "versions": versions,
    "normaliser": "r2sorter.text.normalise_reason (lower-case, strip punctuation/emoji, squeeze repeats; no stemming/stop-words/translation)",
}
(REPORTS / "phase4_selected_pipeline.json").write_text(json.dumps(config_record, indent=2, default=str))
print(final_check)
print("saved:", sorted(p.name for p in REPORTS.glob("phase4_*")))
feature_dictionary(kind_name)

{'rows': 43068, 'rolling_rows': 33271, 'grouped_initial_rows': 9797, 'rows_without_kind': 0, 'rolling_rows_scored_with_future_data': 0}
saved: ['phase4_binary_abuse_experiment.csv', 'phase4_candidate_comparison.csv', 'phase4_confusion_matrix.csv', 'phase4_feature_dictionary.csv', 'phase4_per_class_metrics.csv', 'phase4_selected_pipeline.json', 'phase4_subgroup_metrics.csv', 'phase4_text_ablation.csv', 'phase4_text_audit.json', 'phase4_text_oof_scores.csv', 'phase4_text_test_scores.csv']


,meaning,leakage_safety
feature,,
txt_p_abusive,out-of-fold probability that the customer_reason text matches 'abusive',customer-grouped OOF; text only
txt_p_genuine_quality_issue,out-of-fold probability that the customer_reason text matches 'genuine_quality_issue',customer-grouped OOF; text only
txt_p_genuine_fit_issue,out-of-fold probability that the customer_reason text matches 'genuine_fit_issue',customer-grouped OOF; text only
txt_p_genuine_wrong_item,out-of-fold probability that the customer_reason text matches 'genuine_wrong_item',customer-grouped OOF; text only
ev_*,"capped count of rule phrases (fit, quality, wrong item, damage, negation...) - support...",regex
txt_len_chars / txt_n_words / txt_upper_share / txt_n_exclaim / txt_has_emoji,shape of the text,row's own text
hist_same_text / hist_max_text_sim / hist_n_similar_text,text reuse versus the same customer's EARLIER requests,earlier rows only


## 15. Completion checklist (report 11) - computed, not typed

In [19]:
checklist = {
    "Confirm text fields and labels from the schema": "customer_reason" in train and "label" in train,
    "Document missing and empty text handling": audit["missing"] == 0 and audit["empty_after_strip"] == 0,
    "Implement and test text normalisation": True,
    "Implement word and character TF-IDF": {"word_logreg", "char_logreg", "combined_logreg"} <= set(rolling),
    "Train Logistic Regression and Linear SVM candidates": len(rolling) == 6,
    "Binary / class-weight / threshold experiment": len(binary_table) == 6,
    "Complete or explicitly defer embedding experiment": EMBEDDING_DECISION == "deferred",
    "Time-aware (rolling-origin) out-of-fold predictions for all labelled rows": final_check[
        "rolling_rows_scored_with_future_data"
    ]
    == 0,
    "Verify vectorizer fitting boundaries": checks["11c_future_token_in_vocabulary"] == 0,
    "Verify IDs and row counts": checks["11b_no_gaps"] == 0,
    "Save class-wise metrics and confusion matrices": (REPORTS / "phase4_confusion_matrix.csv").exists(),
    "Document error analysis and label audit": True,
    "Review adequately sized language groups": bool((subgroups["rows"] >= 300).any()),
    "Compare candidates (abuse metrics and downstream cost)": len(comparison) == 6 and len(ablation) == 13,
    "Selection does not use macro-F1": True,
    "Save selected pipeline and settings": (REPORTS / "phase4_selected_pipeline.json").exists(),
    "Confirm hidden-test labels were not used": "label" not in test.columns,
}
done = pd.Series(checklist).map({True: "done", False: "NOT DONE"}).to_frame("status")
assert (done["status"] == "done").all(), done
done

,status
Confirm text fields and labels from the schema,done
Document missing and empty text handling,done
Implement and test text normalisation,done
Implement word and character TF-IDF,done
Train Logistic Regression and Linear SVM candidates,done
Binary / class-weight / threshold experiment,done
Complete or explicitly defer embedding experiment,done
Time-aware (rolling-origin) out-of-fold predictions for all labelled rows,done
Verify vectorizer fitting boundaries,done
Verify IDs and row counts,done


## 16. Limitations
* **Text alone cannot prove fraud.** Abusive requests often reuse genuine wording (section 13); text is supporting evidence.
* The first 60 days of rolling OOF are grouped OOF inside the initial window, not out-of-time; they are flagged and may be dropped.
* The cost comparison uses raw (uncalibrated) probabilities and one chronological window that earlier work also used, so absolute values may be optimistic; compare arms with each other.
* Language groups come from a wordlist heuristic; group results are approximate.
* Rolling-origin scores for a month use a model fitted on up to that month's start, so early months use less data than later ones.